# Exploratory Data Analysis — Copa Oracle 2026
World Cup match data from 1930 to 2022.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

#Style
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

#Load data
combined = pd.read_csv('../data/02-preprocessed/wc_combined.csv')
features = pd.read_csv('../data/03-features/wc_features.csv')

print(f"Combined : {combined.shape[0]} matches | {combined['year'].min()}–{combined['year'].max()}")
print(f"Features : {features.shape[0]} rows × {features.shape[1]} columns")
combined.head()


## 1. Dataset Overview

In [ ]:
print("=" * 50)
print(f"Total matches      : {len(combined)}")
print(f"Years covered      : {combined['year'].min()} – {combined['year'].max()}")
print(f"Unique home teams  : {combined['home_team'].nunique()}")
print(f"Unique away teams  : {combined['away_team'].nunique()}")
print(f"Null values        : {combined.isnull().sum().sum()}")
print("=" * 50)
combined['match_result'].value_counts()


## 2. Match Result Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Overall pie
result_counts = combined['match_result'].value_counts()
labels = {'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'}
axes[0].pie(
    result_counts,
    labels=[labels[r] for r in result_counts.index],
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452', '#55A868'],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Overall Result Distribution (1930–2022)', fontweight='bold')

#By stage
stage_results = combined.groupby(['stage', 'match_result']).size().unstack(fill_value=0)
stage_order   = ['group','round_of_16','quarter_final','semi_final','final']
stage_results = stage_results.reindex([s for s in stage_order if s in stage_results.index])
stage_results.plot(kind='bar', ax=axes[1], color=['#55A868','#DD8452','#4C72B0'],
                   edgecolor='white', width=0.7)
axes[1].set_title('Results by Tournament Stage', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_xticklabels([s.replace('_',' ').title() for s in stage_results.index], rotation=30, ha='right')
axes[1].legend(['Away Win','Draw','Home Win'])

plt.tight_layout()
plt.savefig('../data/04-predictions/eda_result_dist.png', bbox_inches='tight')
plt.show()


## 3. Matches & Goals Per Tournament Year

In [ ]:
yearly = combined.groupby('year').agg(
    matches=('match_result', 'count'),
    total_goals=('home_goals', 'sum'),
    away_goals=('away_goals', 'sum'),
).reset_index()
yearly['total_goals'] += yearly['away_goals']
yearly['goals_per_game'] = (yearly['total_goals'] / yearly['matches']).round(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

#Matches per year
axes[0].bar(yearly['year'], yearly['matches'], color='#4C72B0', width=2.5, edgecolor='white')
axes[0].set_ylabel('Number of Matches')
axes[0].set_title('Matches per World Cup', fontweight='bold')

#Goals per game
axes[1].plot(yearly['year'], yearly['goals_per_game'], marker='o', linewidth=2.5,
             color='#DD8452', markersize=7)
axes[1].axhline(yearly['goals_per_game'].mean(), linestyle='--', color='gray',
                linewidth=1.2, label=f"Mean: {yearly['goals_per_game'].mean():.2f}")
axes[1].set_ylabel('Goals per Game')
axes[1].set_title('Goals per Game Over Time', fontweight='bold')
axes[1].legend()
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig('../data/04-predictions/eda_yearly_stats.png', bbox_inches='tight')
plt.show()
print(yearly[['year','matches','goals_per_game']].to_string(index=False))


## 4. Home Advantage Analysis

In [ ]:
home_adv = combined.groupby('year')['match_result'].apply(
    lambda x: (x == 'H').sum() / len(x) * 100
).reset_index()
home_adv.columns = ['year', 'home_win_pct']

plt.figure(figsize=(14, 4))
plt.bar(home_adv['year'], home_adv['home_win_pct'], color='#4C72B0', width=2.5, edgecolor='white')
plt.axhline(home_adv['home_win_pct'].mean(), color='#DD8452', linestyle='--', linewidth=1.8,
            label=f"Overall avg: {home_adv['home_win_pct'].mean():.1f}%")
plt.title('Home Win % by Tournament Year', fontweight='bold')
plt.ylabel('Home Win %')
plt.xlabel('Year')
plt.legend()
plt.tight_layout()
plt.savefig('../data/04-predictions/eda_home_advantage.png', bbox_inches='tight')
plt.show()

#Host nation effect
host_matches = combined[combined['home_is_host'] == 1]
non_host     = combined[combined['home_is_host'] == 0]
print(f"Host nation home win rate   : {(host_matches['match_result']=='H').mean():.1%}")
print(f"Non-host team home win rate : {(non_host['match_result']=='H').mean():.1%}")


## 5. Top 15 Teams — All-Time World Cup Wins

In [ ]:
wins_home = combined[combined['match_result']=='H'].groupby('home_team').size()
wins_away = combined[combined['match_result']=='A'].groupby('away_team').size()
total_wins = (wins_home.add(wins_away, fill_value=0)).sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 5))
bars = plt.barh(total_wins.index[::-1], total_wins.values[::-1],
                color=plt.cm.Blues_r(np.linspace(0.3, 0.9, 15)), edgecolor='white')
for bar, val in zip(bars, total_wins.values[::-1]):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             str(int(val)), va='center', fontsize=9)
plt.title('Top 15 Teams by All-Time World Cup Wins', fontweight='bold')
plt.xlabel('Total Wins')
plt.tight_layout()
plt.savefig('../data/04-predictions/eda_top_teams.png', bbox_inches='tight')
plt.show()


## 6. Elo Rating Distribution (Final Ratings)

In [ ]:
from src.pipelines.feature_eng_pipeline import ELO_START, ELO_K, DECAY

#Recompute final Elo per team
elo = {}
last_year = {}
def expected(ra, rb): return 1 / (1 + 10**((rb-ra)/400))

for _, row in combined.iterrows():
    ht, at = row['home_team'], row['away_team']
    yr, st = row['year'], row['stage']
    for t in (ht, at):
        if t in last_year and last_year[t] < yr:
            elo[t] = elo.get(t, ELO_START) + DECAY*(ELO_START - elo.get(t, ELO_START))
        last_year[t] = yr
    ra, rb = elo.get(ht, ELO_START), elo.get(at, ELO_START)
    r  = row['match_result']
    sh = 1.0 if r=='H' else (0.5 if r=='D' else 0.0)
    k  = ELO_K.get(st, 32)
    elo[ht] = ra + k*(sh - expected(ra,rb))
    elo[at] = rb + k*((1-sh) - (1-expected(ra,rb)))

elo_df = pd.DataFrame({'team': list(elo.keys()), 'elo': list(elo.values())})
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(elo_df['elo'], bins=30, color='#4C72B0', edgecolor='white')
axes[0].axvline(ELO_START, color='#DD8452', linestyle='--', linewidth=1.5, label='Start (1500)')
axes[0].set_title('Elo Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Elo Rating')
axes[0].legend()

top15 = elo_df.head(15)
axes[1].barh(top15['team'][::-1], top15['elo'][::-1],
             color=plt.cm.Oranges_r(np.linspace(0.3, 0.9, 15)), edgecolor='white')
axes[1].axvline(ELO_START, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Top 15 Teams by Final Elo', fontweight='bold')
axes[1].set_xlabel('Elo Rating')
plt.tight_layout()
plt.savefig('../data/04-predictions/eda_elo.png', bbox_inches='tight')
plt.show()
print(elo_df.head(10).to_string(index=False))


## 7. Feature Correlation Heatmap

In [ ]:
from src.pipelines.feature_eng_pipeline import FEATURE_COLS

corr = features[FEATURE_COLS].corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.4,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('../data/04-predictions/eda_correlation.png', bbox_inches='tight')
plt.show()


## 8. Goals Distribution — Home vs Away

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in zip(
    axes,
    ['home_goals', 'away_goals'],
    ['Home Goals per Match', 'Away Goals per Match'],
    ['#4C72B0', '#DD8452']
):
    counts = combined[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=color, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Goals')
    ax.set_ylabel('Frequency')
    mean_val = combined[col].mean()
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {mean_val:.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/04-predictions/eda_goals_dist.png', bbox_inches='tight')
plt.show()


## 9. Summary Statistics

In [ ]:
print("Match Results")
vc = combined['match_result'].value_counts()
for r, c in vc.items():
    label = {'H':'Home Win','D':'Draw','A':'Away Win'}[r]
    print(f"  {label:<12}: {c:>4}  ({c/len(combined):.1%})")

print()
print("Goals")
print(f"  Avg home goals : {combined['home_goals'].mean():.2f}")
print(f"  Avg away goals : {combined['away_goals'].mean():.2f}")
print(f"  Total goals    : {(combined['home_goals']+combined['away_goals']).sum()}")

print()
print(" Features ")
print(f"  Total features : {len(FEATURE_COLS)}")
print(f"  Any nulls      : {features[FEATURE_COLS].isnull().sum().sum()}")
